In [1]:
from pathlib import Path
import gzip
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [2]:
project = Path("/localdata/qinti/Project/Bee")

gt_file = project / "03_variants/genotype_tables/all_samples.gt_dp_ad.tsv.gz"
sample_file = project / "00_rawdata/metadata/clean_samples.tsv"

out_dir = project / "03_variants/ploidy_check"
out_dir.mkdir(parents=True, exist_ok=True)

samples = pd.read_csv(sample_file, sep="\t")

rows = []
ab_rows = []

with gzip.open(gt_file, "rt") as handle:
    for line in handle:
        chrom, pos, sample, gt, dp, ad = line.rstrip("\n").split("\t")

        if dp == "." or ad == ".":
            continue

        try:
            dp = int(dp)
        except ValueError:
            continue

        ad_parts = ad.split(",")

        if len(ad_parts) != 2:
            continue

        try:
            ref_dp = int(ad_parts[0])
            alt_dp = int(ad_parts[1])
        except ValueError:
            continue

        total_ad = ref_dp + alt_dp

        if dp < 10 or total_ad < 10:
            continue

        ab = alt_dp / total_ad if total_ad > 0 else np.nan

        is_het = gt in {"0/1", "1/0", "0|1", "1|0"}

        rows.append(
            {
                "sample_id": sample,
                "gt": gt,
                "dp": dp,
                "ref_dp": ref_dp,
                "alt_dp": alt_dp,
                "ab": ab,
                "is_het": is_het,
                "central_ab": 0.3 <= ab <= 0.7,
            }
        )

        # Save AB values for plotting, but limit memory by storing only useful fields
        ab_rows.append(
            {
                "sample_id": sample,
                "ab": ab,
                "is_het": is_het,
            }
        )

data = pd.DataFrame(rows)

summary = (
    data.groupby("sample_id")
    .agg(
        n_variant_calls=("gt", "size"),
        n_het_calls=("is_het", "sum"),
        fraction_het_calls=("is_het", "mean"),
        n_central_ab=("central_ab", "sum"),
        fraction_central_ab=("central_ab", "mean"),
        median_depth=("dp", "median"),
        mean_depth=("dp", "mean"),
    )
    .reset_index()
)

summary = summary.merge(
    samples[["sample_id", "family_id", "role"]],
    on="sample_id",
    how="left"
)

summary = summary[
    [
        "sample_id",
        "family_id",
        "role",
        "n_variant_calls",
        "n_het_calls",
        "fraction_het_calls",
        "n_central_ab",
        "fraction_central_ab",
        "median_depth",
        "mean_depth",
    ]
]

summary_file = out_dir / "ploidy_summary.tsv"
summary.to_csv(summary_file, sep="\t", index=False)

print(summary.to_string(index=False))
print(f"\nSaved: {summary_file}")

sample_id family_id      role  n_variant_calls  n_het_calls  fraction_het_calls  n_central_ab  fraction_central_ab  median_depth  mean_depth
     C1_1        C1 offspring           828266        29610            0.035749         10921             0.013185          37.0   37.847984
     C1_2        C1 offspring           828008        29751            0.035931         10683             0.012902          36.0   36.702294
     C1_3        C1 offspring           829796        27371            0.032985          9736             0.011733          33.0   34.045065
     C1_4        C1 offspring           830460        31039            0.037376         11273             0.013574          34.0   35.380212
     C1_5        C1 offspring           828737        30163            0.036396         10888             0.013138          35.0   36.179202
     C1_Q        C1     queen           846162       294103            0.347573        252909             0.298890          32.0   32.851571
     C2_1    

In [4]:
plot_data = summary.sort_values(["family_id", "role", "sample_id"]).copy()
x = np.arange(len(plot_data))

fig, ax = plt.subplots(figsize=(7.2, 2.8), dpi=150)
ax.bar(x, plot_data["n_het_calls"])
ax.set_xticks(x)
ax.set_xticklabels(plot_data["sample_id"], rotation=90)
ax.set_ylabel("Heterozygous SNP calls")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig(out_dir / "heterozygous_snp_counts.pdf")
plt.show()

In [5]:
# Plot 2: fraction central AB
fig, ax = plt.subplots(figsize=(7.2, 2.8), dpi=150)
ax.bar(x, plot_data["fraction_central_ab"])
ax.set_xticks(x)
ax.set_xticklabels(plot_data["sample_id"], rotation=90)
ax.set_ylabel("Fraction of sites with 0.3 ≤ AB ≤ 0.7")
ax.set_xlabel("")
fig.tight_layout()
fig.savefig(out_dir / "central_allele_balance_fraction.pdf")

In [ ]:
# Plot 3: allele balance density for all samples
ab_data = pd.DataFrame(ab_rows)

fig, ax = plt.subplots(figsize=(7.2, 4.8), dpi=150)

for sample_id in plot_data["sample_id"]:
    values = ab_data.loc[
        ab_data["sample_id"] == sample_id,
        "ab"
    ].dropna()

    ax.hist(
        values,
        bins=60,
        range=(0, 1),
        histtype="step",
        density=True,
        linewidth=0.7,
        label=sample_id
    )

ax.set_xlabel("Alternative allele fraction")
ax.set_ylabel("Density")
ax.set_xlim(0, 1)
ax.legend(fontsize=8, frameon=False, ncol=4)
fig.tight_layout()
fig.savefig(out_dir / "allele_balance_density_all_samples.pdf")

In [8]:
# Plot 4: allele balance only for heterozygous calls
het_ab = ab_data.loc[ab_data["is_het"]].copy()

fig, ax = plt.subplots(figsize=(7.2, 4.8), dpi=150)

for sample_id in plot_data["sample_id"]:
    values = het_ab.loc[
        het_ab["sample_id"] == sample_id,
        "ab"
    ].dropna()

    if len(values) == 0:
        continue

    ax.hist(
        values,
        bins=50,
        range=(0, 1),
        histtype="step",
        density=True,
        linewidth=0.8,
        label=sample_id
    )

ax.set_xlabel("Alternative allele fraction at heterozygous calls")
ax.set_ylabel("Density")
ax.set_xlim(0, 1)
ax.legend(fontsize=8, frameon=False, ncol=4)
fig.tight_layout()
fig.savefig(out_dir / "allele_balance_density_heterozygous_calls.pdf")